In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.6 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
dataset_path = "/content/drive/MyDrive/pub_checker/dataset"

## Create YAML File

In [6]:
import yaml

data = {
    "path": "/content/drive/MyDrive/pub_checker/dataset",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "names": {
        0: "NYC_Correct",
        1: "NYC_Incorrect",
        2: "BP_Correct",
        3: "BP_Incorrect",
        4: "SK_Correct",
        5: "SK_Incorrect",
        6: "YORP_Correct",
        7: "YORP_Incorrect",
    }

}
with open("/content/drive/MyDrive/pub_checker/data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml file created.")

data.yaml file created.


## Train YOLO Model

In [7]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
# pre-trained YOLO model
model = YOLO("yolov8s.pt")

In [9]:
import os

val_image_path = "/content/drive/MyDrive/pub_checker/dataset/images/val"

if os.path.exists(val_image_path):
    print(f"Directory '{val_image_path}' exists.")
    print("Contents (first 10 items):")
    contents = os.listdir(val_image_path)
    for item in contents[:10]: # Print only first 10 items to avoid long output
        print(item)
    if not contents:
        print("The directory is empty.")
else:
    print(f"Directory '{val_image_path}' does NOT exist.")

# Also check the parent dataset path for overall structure verification
dataset_root_path = "/content/drive/MyDrive/pub_checker/dataset"
if os.path.exists(dataset_root_path):
    print(f"\nDirectory '{dataset_root_path}' exists.")
    print("Contents (first 10 items):")
    contents = os.listdir(dataset_root_path)
    for item in contents[:10]: # Print only first 10 items
        print(item)
else:
    print(f"\nDirectory '{dataset_root_path}' does NOT exist.")

Directory '/content/drive/MyDrive/pub_checker/dataset/images/val' exists.
Contents (first 10 items):
image_1175.jpg
image_1165.jpg
image_1160.jpg
image_1173.jpg
image_1156.jpg
image_1145.jpg
image_1147.jpg
image_1152.jpg
image_1112.jpg
image_1137.jpg

Directory '/content/drive/MyDrive/pub_checker/dataset' exists.
Contents (first 10 items):
images
labels


In [10]:
# first training
model.train(
    data="/content/drive/MyDrive/pub_checker/data.yaml",
    project="/content/drive/MyDrive/pub_checker/runs",
    name="logo_model_v1",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=15,
    device=0,
    workers=2,
    cache=True,
    amp=True
)

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/pub_checker/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=logo_model_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fd3743dc530>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [15]:
best_model_v1 = YOLO("/content/drive/MyDrive/pub_checker/runs/logo_model_v1/weights/best.pt")

In [ ]:
# # longer training (epoch 200, patience 30)
# model.train(
#     data="/content/drive/MyDrive/pub_checker/data.yaml",
#     project="/content/drive/MyDrive/pub_checker/runs",
#     name="logo_model_v2",
#     epochs=200,
#     imgsz=800,
#     batch=8,
#     patience=30,
#     device=0,
#     workers=2,
#     cache=True,
#     amp=True
# )

## Metrics

In [16]:
print("Metrics for Pre-trained YOLOv8 model")
metrics = best_model_v1.val(
    data="/content/drive/MyDrive/pub_checker/data.yaml",
    plots=True
)



Metrics for Pre-trained YOLOv8 model
Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 25.7±54.9 ms, read: 14.6±14.3 MB/s, size: 186.3 KB)
val: Scanning /content/drive/MyDrive/pub_checker/dataset/labels/val.cache... 242 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 242/242 67.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.2it/s 7.2s
                   all        242        766      0.958      0.985      0.992      0.974
           NYC_Correct         81         81       0.92          1      0.987       0.98
         NYC_Incorrect        121        121      0.992      0.974      0.994       0.98
            BP_Correct         70         70      0.908       0.99      0.991      0.986
          BP_Incorrect        142        142      0.992      0.979      0.9

In [17]:
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

mAP50: 0.9919
mAP50-95: 0.9735
Precision: 0.9576
Recall: 0.9852
